<a href="https://colab.research.google.com/github/stefisha/genai24/blob/main/Report_Rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install openai pdfplumber langchain-community tiktoken chromadb

In [ ]:
import os
import pdfplumber
import logging
import openai
from google.colab import userdata
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import Chroma
from langchain.chat_models import ChatOpenAI
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.prompts import PromptTemplate

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

In [ ]:
from dotenv import load_dotenv
load_dotenv()
# OPENAI_API_KEY is read from your environment; never commit credentials.


In [ ]:
openai.api_key = userdata.get('OPENAI_API_KEY')

In [ ]:
# Configure the logger
logger = logging.getLogger("PDF_RAG")
logger.setLevel(logging.INFO)

# Remove existing handlers to avoid duplication
if logger.hasHandlers():
    logger.handlers.clear()

# Create console handler
console_handler = logging.StreamHandler()
# file_handler = logging.FileHandler("/content/script.log")

# Define a formatter
formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
console_handler.setFormatter(formatter)
# file_handler.setFormatter(formatter)

# Add the console handler to the logger
logger.addHandler(console_handler)

# Debugging log handlers
logger.info(f"Logger configured with {len(logger.handlers)} handler(s).")


In [ ]:
# Step 1: Extract text from PDF
def extract_text_from_pdf(pdf_path):
    """Extract text from a PDF file."""
    logger.info(f"Starting text extraction from: {pdf_path}")
    try:
        with pdfplumber.open(pdf_path) as pdf:
            text = ""
            for page in pdf.pages:
                text += page.extract_text() + "\n"
        logger.info("Text extraction completed.")
        return text
    except Exception as e:
        logger.error(f"Error during text extraction: {e}")
        raise

In [ ]:
# Step 2: Embed text
def embed_text(text, embedding_model):
    """Embed text using the specified embedding model."""
    logger.info("Starting text embedding.")
    try:
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
        chunks = text_splitter.split_text(text)
        embeddings = embedding_model.embed_documents(chunks)
        logger.info("Text embedding completed.")
        return chunks, embeddings
    except Exception as e:
        logger.error(f"Error during text embedding: {e}")
        raise

In [ ]:
# Step 3: Store embeddings in Chroma
def store_embeddings(chunks, embeddings, persist_directory="chroma_db"):
    """Store embeddings in Chroma vector store."""
    logger.info("Starting to store embeddings in Chroma.")
    try:
        vector_store = Chroma.from_texts(
            texts=chunks,
            embedding=embedding_model,
            persist_directory=persist_directory
        )
        vector_store.persist()
        logger.info("Embeddings successfully stored in Chroma.")
        return vector_store
    except Exception as e:
        logger.error(f"Error during storing embeddings: {e}")
        raise

In [ ]:
# Step 4: Query the Vector Store
def query_vector_store(query, vector_store, chat_model):
    """Query the vector store and generate a response."""
    logger.info("Starting query to the vector store.")
    try:
        retriever = vector_store.as_retriever()
        relevant_docs = retriever.get_relevant_documents(query)
        context = "\n".join([doc.page_content for doc in relevant_docs])

        prompt = PromptTemplate(
            input_variables=["context", "query"],
            template="Given the following context, answer the user's query:\n\nContext:\n{context}\n\nQuery:\n{query}"
        )
        formatted_prompt = prompt.format(context=context, query=query)
        response = chat_model.predict(formatted_prompt)
        logger.info("Query successfully processed.")
        return response
    except Exception as e:
        logger.error(f"Error during query processing: {e}")
        raise

In [ ]:
# Main Workflow
if __name__ == "__main__":
    try:
        logger.info("Main workflow started.")

        # Specify the PDF file path
        # pdf_path = input("Please enter the path to the PDF file: ").strip()
        pdf_path = "/content/drive/MyDrive/PDF_Rag/scammer-agent.pdf"
        logger.info(f"Processing file: {pdf_path}")

        # Step 1: Extract text from PDF
        text = extract_text_from_pdf(pdf_path)
        logger.info("Step 1: Text extraction completed.")

        # Step 2: Initialize embedding model and embed text
        embedding_model = OpenAIEmbeddings()  # Requires OpenAI API key
        chunks, embeddings = embed_text(text, embedding_model)
        logger.info("Step 2: Text embedding completed.")

        # Step 3: Store embeddings in Chroma
        vector_store = store_embeddings(chunks, embeddings)
        logger.info("Step 3: Embeddings stored successfully.")

        # Step 4: Query and retrieve relevant context
        chat_model = ChatOpenAI(model_name="gpt-4", temperature=0)  # Requires OpenAI API key
        # query = input("Please enter your query: ").strip()query = input("Please enter your query: ").strip()
        query = "What is the document about?"  # Example query
        response = query_vector_store(query, vector_store, chat_model)
        logger.info("Step 4: Query successfully processed.")

        # Print the response
        logger.info(f"Response: {response}")

    except Exception as e:
        logger.error(f"An error occurred in the main workflow: {e}")
        # Flush all logs before exiting
        for handler in logger.handlers:
            handler.flush()